<a href="https://colab.research.google.com/github/PALAK7890/Time_value/blob/main/Time_Value.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [73]:
import pandas as pd
import numpy as np
events = pd.read_csv("startup_realistic_event_log.csv")
users = pd.read_csv("startup_realistic_user_summary.csv")

In [57]:
events["event_time"] = pd.to_datetime(events["event_time"])
users["signup_time"] = pd.to_datetime(users["signup_time"])
users["first_value_time"] = pd.to_datetime(users["first_value_time"])
users["last_active_time"] = pd.to_datetime(users["last_active_time"])

In [58]:
events = events.drop_duplicates()
users = users.drop_duplicates()

In [59]:
events["event_name"] = events["event_name"].str.lower().str.strip()
events["event_category"] = events["event_category"].str.lower().str.strip()

In [74]:
def ttv_bucket(x):
    if x == -1:
        return "Never"
    elif x <= 1:
        return "0-1 hr"
    elif x <= 24:
        return "1-24 hr"
    elif x <= 72:
        return "1-3 days"
    else:
        return "3+ days"

users["ttv_bucket"] = users["time_to_value_hours"].apply(ttv_bucket)

In [61]:
users["engagement_score"] = (
    users["sessions"] * 0.4 +
    users["feature_breadth"] * 0.3 +
    users["stickiness_ratio"] * 0.3
)

In [62]:
users["early_risk"] = (users["early_48h_events"] < 3).astype(int)

In [64]:
users["ttv_efficiency_score"] = users["time_to_value_hours"].apply(
    lambda x: 0 if x == -1 else 1 / (1 + x)
)
users["activation_quality"] = (
    users["reached_value"].astype(int) * 0.5 +
    users["ttv_efficiency_score"] * 0.5
)

In [67]:
users["friction_impact"] = (
    users["friction_rate"] * 0.6 +
    users["onboarding_friction_score"] * 0.4
)
users["power_user_score"] = (
    users["engagement_score"] * 0.5 +
    users["activation_quality"] * 0.3 -
    users["friction_impact"] * 0.2
)

In [68]:
users["user_segment"] = pd.qcut(
    users["power_user_score"],
    q=3,
    labels=["Low Value", "Mid Value", "High Value"]
)

In [69]:
from sklearn.model_selection import train_test_split

features = [
    "time_to_value_hours",
    "sessions",
    "feature_breadth",
    "stickiness_ratio",
    "friction_rate",
    "early_48h_events",
    "engagement_score",
    "activation_quality"
]

X = users[features]
y = users["churned"]

X = X.fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [70]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.94      0.92      0.93       450
        True       0.96      0.97      0.97       950

    accuracy                           0.96      1400
   macro avg       0.95      0.95      0.95      1400
weighted avg       0.96      0.96      0.96      1400



In [71]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=features)
print(importance.sort_values(ascending=False))

sessions               0.423427
engagement_score       0.291121
friction_rate          0.092219
stickiness_ratio       0.091707
time_to_value_hours    0.034594
activation_quality     0.034250
early_48h_events       0.016638
feature_breadth        0.016045
dtype: float64


In [75]:
users["ttv_clean"] = users["time_to_value_hours"].fillna(999)

users["activation_velocity_index"] = np.where(
    users["reached_value"] == True,
    1 / (1 + users["ttv_clean"]),
    0
)

In [76]:
value_events = [
    "create_project",
    "connect_integration",
    "invite_team_member",
    "export_report",
    "publish_dashboard"
]

value_depth = (
    events[events["event_name"].isin(value_events)]
    .groupby("user_id")["event_name"]
    .nunique()
    .reset_index(name="value_depth_score")
)

users = users.merge(value_depth, on="user_id", how="left")
users["value_depth_score"] = users["value_depth_score"].fillna(0)

In [77]:
friction_events = events[events["event_category"] == "friction"]

first_friction = (
    friction_events.groupby("user_id")["event_time"]
    .min()
    .reset_index(name="first_friction_time")
)

users = users.merge(first_friction, on="user_id", how="left")

events_friction = events.merge(
    users[["user_id", "first_friction_time"]],
    on="user_id",
    how="left"
)

post_friction_events = events_friction[
    (events_friction["first_friction_time"].notna()) &
    (events_friction["event_time"] > events_friction["first_friction_time"])
]

recovery = (
    post_friction_events.groupby("user_id")
    .size()
    .reset_index(name="post_friction_activity")
)

users = users.merge(recovery, on="user_id", how="left")
users["post_friction_activity"] = users["post_friction_activity"].fillna(0)

users["friction_recovery_score"] = np.where(
    users["friction_events"] > 0,
    users["post_friction_activity"] / users["friction_events"],
    0
)

In [78]:
daily_activity = (
    events.groupby(["user_id", "event_date"])
    .size()
    .reset_index(name="daily_events")
)

active_days_count = (
    daily_activity.groupby("user_id")["event_date"]
    .nunique()
    .reset_index(name="active_days_count")
)

users = users.merge(active_days_count, on="user_id", how="left")
users["active_days_count"] = users["active_days_count"].fillna(0)

users["habit_formation_score"] = (
    users["active_days_count"] / users["observed_lifetime_days"]
).clip(0, 1)